# 3.3 · 提示（Prompts）与参数补全

**Prompts and argument completion**

> **学完本课你能：** 发布由用户调用的提示模板，支持参数以及多消息、富含资源的结果，并为提示模板和资源模板加入参数自动补全。

官方阅读材料：[Spec: Prompts](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts) · [Spec: Completion](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/completion)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个带三种提示模板的服务器
`@server.prompt()` 把一个函数变成提示模板（Prompt）：函数参数成为提示参数（没有默认值的即为必需，用 `Field(description=...)` 添加描述），文档字符串成为描述。函数可以返回一个字符串（一条用户消息），也可以返回由 `UserMessage` / `AssistantMessage` 和原始内容块组成的列表。

In [ ]:
import json
import logging
from typing import Annotated

from pydantic import Field

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import AssistantMessage, UserMessage
from mcp.shared.exceptions import MCPError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1800):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


STYLE_GUIDE = "1. Type hints everywhere.\n2. No bare except.\n3. Functions under 40 lines."

server = MCPServer("dev-workflows")


@server.resource("docs://style-guide", mime_type="text/markdown")
def style_guide() -> str:
    return STYLE_GUIDE


@server.prompt(title="Request Code Review")
def code_review(code: Annotated[str, Field(description="The code to review")],
                language: Annotated[str, Field(description="Programming language")] = "python") -> str:
    """Asks the LLM to analyze code quality and suggest improvements."""
    return f"Please review this {language} code:\n{code}"


@server.prompt(title="Plan a vacation")
def plan_vacation(destination: Annotated[str, Field(description="City or region")],
                  days: Annotated[int, Field(description="Trip length in days (whole number)")] = 7) -> list:
    """Guide the model through planning a trip, with a worked example of the expected format."""
    return [
        UserMessage("Plan a 2-day trip to Lisbon."),
        AssistantMessage("Day 1: Alfama + tram 28. Day 2: Belém + LX Factory. Budget: ~€180."),
        UserMessage(f"Great format. Now plan a {days}-day trip to {destination}. "
                    "Check my calendar resource and the weather tool before proposing dates."),
    ]


@server.prompt(title="Review against the style guide")
def review_with_guide(code: Annotated[str, Field(description="The code to review")]) -> list:
    """Review code against the team style guide, which is embedded in the conversation."""
    return [
        UserMessage(types.EmbeddedResource(type="resource", resource=types.TextResourceContents(
            uri="docs://style-guide", mime_type="text/markdown", text=STYLE_GUIDE))),
        UserMessage(f"Review this code strictly against the style guide above:\n{code}"),
    ]


print("prompts:", [p.name for p in await server.list_prompts()])

### 2. `prompts/list`：斜杠命令菜单需要什么
参数携带 `name`、`description` 和 `required`——但没有类型。宿主（Host）会把它渲染成 `/code_review <code> [language]`。

In [ ]:
async with Client(server) as client:
    listing = await client.list_prompts()
    print("capabilities.prompts:", client.server_capabilities.prompts)
    show(listing.prompts[0])
    print()
    for p in listing.prompts:
        args = " ".join(f"<{a.name}>" if a.required else f"[{a.name}]" for a in p.arguments or [])
        print(f"/{p.name:<18} {args:<22} {p.title}")

### 3. `prompts/get`：带参数渲染
参数以字符串形式传输；`MCPServer` 会依据类型注解把 `days` 的 `"5"` 转换为 `int`。留意多消息提示模板（Prompt）中的角色，以及嵌入式资源（Embedded Resource）块。

In [ ]:
async with Client(server) as client:
    print("--- code_review")
    show(await client.get_prompt("code_review", {"code": "def hello():\n    print('world')"}))
    print("\n--- plan_vacation (days arrives as the string '5')")
    for m in (await client.get_prompt("plan_vacation", {"destination": "Barcelona", "days": "5"})).messages:
        print(f"{m.role:>9}: {m.content.text}")
    print("\n--- review_with_guide")
    show(await client.get_prompt("review_with_guide", {"code": "try:\n    run()\nexcept:\n    pass"}))

### 4. 宿主的斜杠命令处理器
是宿主（Host）而不是协议，负责把 `PromptMessage` 转换成其模型的消息格式。这里：解析 `/command key=value`，调用 `prompts/get`，再转换成 Anthropic 风格的消息（嵌入式资源变成带标签的文本；相邻的同角色消息会被合并，因为许多聊天 API 要求角色交替出现）。

In [ ]:
import shlex


def to_chat_messages(prompt_messages):
    chat = []
    for m in prompt_messages:
        c = m.content
        if c.type == "text":
            block = {"type": "text", "text": c.text}
        elif c.type == "resource":
            body = getattr(c.resource, "text", "<binary>")
            block = {"type": "text", "text": f"<resource uri='{c.resource.uri}'>\n{body}\n</resource>"}
        elif c.type == "image":
            block = {"type": "image", "source": {"type": "base64", "media_type": c.mime_type, "data": c.data}}
        else:
            block = {"type": "text", "text": f"[{c.type}]"}
        if chat and chat[-1]["role"] == m.role:
            chat[-1]["content"].append(block)
        else:
            chat.append({"role": m.role, "content": [block]})
    return chat


async def run_slash_command(client, line: str):
    name, *pairs = shlex.split(line.lstrip("/"))
    arguments = dict(pair.split("=", 1) for pair in pairs)
    result = await client.get_prompt(name, arguments)
    return to_chat_messages(result.messages)


async with Client(server) as client:
    messages = await run_slash_command(client, "/review_with_guide code='except: pass'")
print(json.dumps(messages, indent=2))

### 5. 动手破坏：错误参数与错误码
规范规定，未知提示模板名称和缺失必需参数应当（SHOULD）返回 `-32602`。来看看 SDK 实际发送的是什么。

In [ ]:
async with Client(server) as client:
    for label, name, args in [("missing required arg", "code_review", {}),
                              ("unknown prompt", "deploy", {}),
                              ("non-numeric days", "plan_vacation", {"destination": "Rome", "days": "a week"})]:
        try:
            await client.get_prompt(name, args)
        except MCPError as err:
            print(f"{label:<22} -> code={err.code} message={err.message!r}")

三种情况返回的都是 `-32603` “Internal server error”——这是 SDK 的一个缺口，而且对正在填写表单的用户毫无帮助。在处理函数中修正它：接收原始字符串，自行校验，并抛出带 `INVALID_PARAMS` 的 `MCPError`，SDK 会原样透传它。

In [ ]:
from mcp_types import INVALID_PARAMS


@server.prompt(title="Plan a vacation (validated)")
def plan_trip(destination: Annotated[str, Field(description="City or region")],
              days: Annotated[str, Field(description="Trip length in days, 1-30")] = "7") -> str:
    """Plan a trip; validates its arguments and reports problems as -32602."""
    if not days.isdigit() or not 1 <= int(days) <= 30:
        raise MCPError(INVALID_PARAMS, f"Argument 'days' must be a whole number from 1 to 30, got {days!r}",
                       data={"argument": "days"})
    return f"Plan a {int(days)}-day trip to {destination}."


async with Client(server) as client:
    try:
        await client.get_prompt("plan_trip", {"destination": "Rome", "days": "a week"})
    except MCPError as err:
        print("fixed:", err.code, err.message, err.data)

### 6. 为提示参数和模板变量提供参数补全
一个 `@server.completion()` 处理函数服务所有 `completion/complete` 请求；它接收 `ref`、`argument`（`name`、`value`），以及包含已确定参数的 `context`。注册它之后，服务器就会公布 `completions` 能力（Capability）。我们把值的数量上限设为 100，并报告 `total` / `hasMore`。

In [ ]:
LANGUAGES = ["python", "pytorch", "pyside", "perl", "php", "javascript", "java", "go", "rust"]
FRAMEWORKS = {"python": ["flask", "fastapi", "django"], "javascript": ["express", "fastify", "next"]}
CITIES = ["Paris", "Park City", "Parma", "Barcelona", "Barbados", "Berlin"]


@server.resource("weather://forecast/{city}/{date}", mime_type="application/json")
def forecast(city: str, date: str) -> str:
    return json.dumps({"city": city, "date": date, "high_c": 27})


@server.prompt()
def scaffold(language: str, framework: str) -> str:
    """Scaffold a new web service."""
    return f"Create a {framework} service in {language}."


def ranked(options, prefix, limit=100):
    matches = [o for o in options if o.lower().startswith(prefix.lower())]
    return types.Completion(values=matches[:limit], total=len(matches), has_more=len(matches) > limit)


@server.completion()
async def complete(ref, argument, context):
    resolved = (context.arguments if context and context.arguments else {})
    if isinstance(ref, types.PromptReference):
        if argument.name == "language":
            return ranked(LANGUAGES, argument.value)
        if argument.name == "framework":
            return ranked(FRAMEWORKS.get(resolved.get("language", ""), []), argument.value)
    if isinstance(ref, types.ResourceTemplateReference) and ref.uri == "weather://forecast/{city}/{date}":
        if argument.name == "city":
            return ranked(CITIES, argument.value)
    return None   # no suggestions -> empty values


prompt_ref = types.PromptReference(type="ref/prompt", name="scaffold")
template_ref = types.ResourceTemplateReference(type="ref/resource", uri="weather://forecast/{city}/{date}")

async with Client(server) as client:
    print("completions capability declared:", client.server_capabilities.completions is not None)
    show(await client.complete(prompt_ref, {"name": "language", "value": "py"}))
    print("framework 'f' with language=python:", (await client.complete(
        prompt_ref, {"name": "framework", "value": "f"}, {"language": "python"})).completion.values)
    print("framework 'f' with language=javascript:", (await client.complete(
        prompt_ref, {"name": "framework", "value": "f"}, {"language": "javascript"})).completion.values)
    print("city 'Par' for the forecast template:", (await client.complete(
        template_ref, {"name": "city", "value": "Par"})).completion.values)

### 7. 线路上的参数补全，以及一个不具备该能力的服务器
先看通过 stdio 进行的原始 `completion/complete` 交换，然后把同样的请求发给一个从未注册补全处理函数的服务器：得到 `-32601` 方法未找到，与规范所列完全一致。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "complete_server.py").write_text(textwrap.dedent('''
    import sys
    import mcp_types as types
    from mcp.server import MCPServer
    server = MCPServer("completer")

    @server.prompt()
    def code_review(code: str, language: str) -> str:
        """Review code."""
        return f"Review this {language} code: {code}"

    if "--with-completion" in sys.argv:
        @server.completion()
        async def complete(ref, argument, context):
            langs = ["python", "pytorch", "pyside", "perl", "php"]
            hits = [l for l in langs if l.startswith(argument.value)]
            return types.Completion(values=hits[:3], total=len(hits), has_more=len(hits) > 3)

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}
request = {"jsonrpc": "2.0", "id": 1, "method": "completion/complete",
           "params": {"ref": {"type": "ref/prompt", "name": "code_review"},
                      "argument": {"name": "language", "value": "p"}, "_meta": META}}

for flags in (["--with-completion"], []):
    proc = subprocess.Popen([sys.executable, str(workdir / "complete_server.py"), *flags], stdin=subprocess.PIPE,
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    proc.stdin.write(json.dumps(request) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)
    print(f"server flags={flags}\n>>> {json.dumps({k: v for k, v in request['params'].items() if k != '_meta'})}")
    print("<<<", json.dumps(reply), "\n")
    proc.stdin.close()
    proc.wait(timeout=10)

### 8. 客户端防抖
客户端应当（SHOULD）做防抖（Debounce）。模拟用户快速输入 `b`、`ba`、`bar`、`barc`，然后停顿：只有最后一个值会被发送出去。

In [ ]:
import asyncio

sent = []


async def debounced_complete(client, ref, arg_name, keystrokes, delay=0.15):
    pending = None

    async def fire(value):
        await asyncio.sleep(delay)
        sent.append(value)
        return (await client.complete(ref, {"name": arg_name, "value": value})).completion.values

    for value, gap in keystrokes:
        if pending:
            pending.cancel()
        pending = asyncio.create_task(fire(value))
        await asyncio.sleep(gap)
    return await pending


async with Client(server) as client:
    typing = [("b", 0.03), ("ba", 0.03), ("bar", 0.03), ("barc", 0.0)]
    values = await debounced_complete(client, template_ref, "city", typing)
print("keystrokes:", [k for k, _ in typing], "| requests actually sent:", sent, "| suggestions:", values)

### 动手练习
1. 添加一个提示模板（Prompt）`/summarize_meetings`，带有 `date` 参数和 `audience` 参数（`team` 或 `exec`），返回一条用户消息外加一条设定格式的 `AssistantMessage`。为 `audience` 添加补全。
2. 让 `review_with_guide` 返回一个指向 `docs://style-guide` 的 `types.ResourceLink`，而不是嵌入它。更新 `to_chat_messages`，让宿主在调用模型之前先用 `resources/read` <em>获取</em>被链接的资源。两种做法各自在什么时候更好？
3. 修改 `ranked`，让它在没有前缀匹配时退而使用子串（近似模糊）匹配，并让城市补全最多返回 2 个值。检查 `total` 和 `hasMore`。
4. 信息泄露：给补全处理函数一份“私有项目”列表，让它只返回调用方有权看到的项目。在真实的服务器中，调用方的身份应该从哪里来？（提示：第 6.1 课。）